
# 03 — Experimental Validation: Full Scorecard & Pull Distributions

This notebook validates SDGFT predictions against **22 precision measurements** from:
- **PDG 2024** (particle masses, couplings, CKM matrix)
- **Planck 2018** (cosmological parameters)
- **NuFIT 5.3** (neutrino mixing angles)
- **BICEP/Keck 2021** (tensor-to-scalar ratio)

We compute the physics scorecard using both the **analytic formulas** and the **GNN surrogate**,
then visualize pull distributions and per-category breakdowns.

> **Oracle Database DOI:** [10.5281/zenodo.18863347](https://doi.org/10.5281/zenodo.18863347)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from sdgft_ml.validation import (
    EXPERIMENTAL_DATA,
    validate_at_axiom,
    validate_at_point,
    chi_squared,
    scorecard,
)
from sdgft_ml.inference import SDGFTPredictor

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Analytic Validation at the Axiom Point

In [ ]:
# Full scorecard from the analytic ParametricForward
results = validate_at_axiom()
scorecard(results)

In [ ]:
# χ² summary
chi2_info = chi_squared(results)
print(f"\n{'='*50}")
print(f"  Total χ² = {chi2_info['chi2']:.2f}")
print(f"  N_dof    = {chi2_info['ndof']}")
print(f"  χ²/dof   = {chi2_info['chi2_per_dof']:.3f}")
print(f"  p-value  = {chi2_info['p_value']:.4f}")
print(f"{'='*50}")

## 2. Pull Distribution

In [ ]:
# Extract pulls
names = []
pulls = []
categories = []
statuses = []

for key, r in results.items():
    names.append(r["name"])
    pulls.append(r["pull"])
    categories.append(r["category"])
    statuses.append(r["status"])

df_pulls = pd.DataFrame({
    "name": names,
    "pull": pulls,
    "category": categories,
    "status": statuses,
    "abs_pull": np.abs(pulls),
}).sort_values("abs_pull")

In [ ]:
# Horizontal bar chart of pulls
cat_colors = {
    "cosmology": "#2196F3",
    "inflation": "#FF9800",
    "particle": "#4CAF50",
    "gravity": "#9C27B0",
}

fig, ax = plt.subplots(figsize=(10, 10))
y_pos = np.arange(len(df_pulls))
colors = [cat_colors.get(c, "gray") for c in df_pulls["category"]]

bars = ax.barh(y_pos, df_pulls["pull"].values, color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_pulls["name"].values, fontsize=9)

# Bands
ax.axvspan(-1, 1, alpha=0.1, color="green", label="<1σ (excellent)")
ax.axvspan(-2, -1, alpha=0.05, color="blue")
ax.axvspan(1, 2, alpha=0.05, color="blue", label="1-2σ (good)")
ax.axvspan(-3, -2, alpha=0.05, color="orange")
ax.axvspan(2, 3, alpha=0.05, color="orange", label="2-3σ (tension)")
ax.axvline(0, color="black", lw=0.5)

ax.set_xlabel("Pull (σ)")
ax.set_title("SDGFT Pull Distribution — Axiom Point (Δ=5/24, δ_g=1/24)")
ax.legend(loc="lower right", fontsize=9)

# Legend for categories
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, label=cat) for cat, c in cat_colors.items()]
ax.legend(handles=handles, loc="upper right", fontsize=9, title="Category")

plt.tight_layout()
plt.show()

In [ ]:
# Pull histogram — should be approximately Gaussian if model is correct
from scipy import stats

pull_values = df_pulls["pull"].values

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pull_values, bins=15, density=True, alpha=0.7, color="steelblue", edgecolor="white")

# Overlay standard normal
x = np.linspace(-4, 4, 200)
ax.plot(x, stats.norm.pdf(x), 'r-', lw=2, label="N(0,1)")

mean_pull = np.mean(pull_values)
std_pull = np.std(pull_values)
ax.set_title(f"Pull Distribution (mean={mean_pull:.2f}, std={std_pull:.2f})")
ax.set_xlabel("Pull (σ)")
ax.set_ylabel("Density")
ax.legend()

plt.tight_layout()
plt.show()

## 3. Per-Category Breakdown

In [ ]:
# Per-category χ²
per_cat = chi2_info["per_category"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of χ²/dof per category
cats = list(per_cat.keys())
chi2_dofs = [per_cat[c]["chi2_per_dof"] for c in cats]
ndofs = [per_cat[c]["ndof"] for c in cats]

colors = [cat_colors.get(c, "gray") for c in cats]
bars = axes[0].bar(cats, chi2_dofs, color=colors, alpha=0.8)
axes[0].axhline(1.0, color='red', ls='--', alpha=0.7, label='χ²/dof = 1')
axes[0].set_ylabel("χ²/dof")
axes[0].set_title("χ² per Category")
axes[0].legend()

# Add ndof labels
for bar, n in zip(bars, ndofs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f"n={n}", ha='center', fontsize=9)

# Pie chart of status counts
status_counts = df_pulls["status"].value_counts()
status_colors = {
    "excellent": "#4CAF50",
    "good": "#8BC34A",
    "tension": "#FF9800",
    "FAIL": "#F44336",
}
labels = status_counts.index
axes[1].pie(
    status_counts.values,
    labels=[f"{l} ({v})" for l, v in zip(labels, status_counts.values)],
    colors=[status_colors.get(l, "gray") for l in labels],
    autopct='%1.0f%%',
    startangle=90,
)
axes[1].set_title(f"Status Distribution (N={len(results)})")

plt.tight_layout()
plt.show()

## 4. GNN Surrogate vs Analytic Comparison

Compare the GNN ensemble predictions to the exact analytic formulas at the axiom point.

In [ ]:
predictor = SDGFTPredictor()
gnn_preds = predictor.predict_with_uncertainty()

In [ ]:
# Build comparison table: analytic vs GNN vs experiment
rows = []
for key, r in results.items():
    if key in gnn_preds:
        gnn = gnn_preds[key]
        exp = EXPERIMENTAL_DATA[key]
        eff_sigma = max(exp.sigma, exp.theory_sigma)
        gnn_pull = (gnn["mean"] - exp.value) / eff_sigma if eff_sigma > 0 else 0
        rows.append({
            "Observable": r["name"],
            "Experiment": exp.value,
            "Analytic": r["theory"],
            "GNN (mean)": gnn["mean"],
            "GNN (std)": gnn["std"],
            "Pull (analytic)": r["pull"],
            "Pull (GNN)": gnn_pull,
            "|Δ GNN-Analytic|": abs(gnn["mean"] - r["theory"]),
        })

df_compare = pd.DataFrame(rows)
display(df_compare.style.format({
    "Experiment": "{:.6g}", "Analytic": "{:.6g}",
    "GNN (mean)": "{:.6g}", "GNN (std)": "{:.4g}",
    "Pull (analytic)": "{:+.2f}", "Pull (GNN)": "{:+.2f}",
    "|Δ GNN-Analytic|": "{:.4g}",
}))

In [ ]:
# Scatter: analytic pull vs GNN pull (exclude extreme outliers for readability)
df_plot = df_compare[df_compare["Pull (GNN)"].abs() < 50].copy()

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df_plot["Pull (analytic)"], df_plot["Pull (GNN)"],
           s=60, alpha=0.8, edgecolors="black", lw=0.5)

for _, row in df_plot.iterrows():
    ax.annotate(row["Observable"][:12], 
                (row["Pull (analytic)"], row["Pull (GNN)"]),
                fontsize=7, alpha=0.7)

lim = max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1]),
          abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1]))
ax.plot([-lim, lim], [-lim, lim], 'r--', alpha=0.5, label='y=x (perfect agreement)')
ax.set_xlabel("Pull (Analytic)")
ax.set_ylabel("Pull (GNN Surrogate)")
ax.set_title("Analytic vs GNN Surrogate at Axiom Point")
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

n_excluded = len(df_compare) - len(df_plot)
if n_excluded:
    ax.text(0.02, 0.02, f"({n_excluded} outlier(s) excluded: |pull| > 50)",
            transform=ax.transAxes, fontsize=8, alpha=0.6)

plt.tight_layout()
plt.show()

## 5. Validation at Off-Axiom Points

How does the scorecard change if we perturb Δ or δ_g slightly from the axiom?

In [ ]:
# Scan χ² along Δ at fixed δ_g = 1/24
deltas = np.linspace(0.19, 0.23, 100)
chi2_scan = []

for d in deltas:
    r = validate_at_point(d, 1/24)
    c = chi_squared(r)
    chi2_scan.append(c["chi2_per_dof"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(deltas, chi2_scan, 'b-', lw=2)
ax.axvline(5/24, color='r', ls='--', label=f'Axiom Δ=5/24≈{5/24:.4f}')
ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel("Δ")
ax.set_ylabel("χ²/dof")
ax.set_title("Analytic χ²/dof vs Δ (δ_g = 1/24 fixed)")
ax.legend()
ax.set_ylim(0, max(chi2_scan) * 1.1)

plt.tight_layout()
plt.show()

## 6. Detailed Observable Table

In [ ]:
# Full experimental data reference
rows = []
for key, exp in EXPERIMENTAL_DATA.items():
    rows.append({
        "Key": key,
        "Name": exp.name,
        "Value": exp.value,
        "σ (exp)": exp.sigma,
        "σ (theory)": exp.theory_sigma,
        "σ (eff)": max(exp.sigma, exp.theory_sigma),
        "Unit": exp.unit,
        "Source": exp.source,
        "Category": exp.category,
    })

df_exp = pd.DataFrame(rows)
display(df_exp.style.format({
    "Value": "{:.6g}", "σ (exp)": "{:.4g}",
    "σ (theory)": "{:.4g}", "σ (eff)": "{:.4g}",
}))

---

**Next:** [04 Predictions & Frontier](04_predictions_frontier.ipynb) — Novel predictions: W-boson mass, muon g-2, gravitational waves